In [18]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [19]:
load_dotenv()

True

In [20]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [21]:
llm=ChatOpenAI()

def chat_node(state:ChatState):

    #take user message
    message=  state['messages']

    #send to llm
    response = llm.invoke(message)

    #
    return {'messages':[response]}

In [22]:
checkpointer=MemorySaver()
graph = StateGraph(ChatState)

#add nodes
graph.add_node('chat_node',chat_node)

#add edges
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot=graph.compile(checkpointer=checkpointer)

In [23]:
thread_id='1'

while True:

    user_message = input('Type Here: ')

    print("User - ", user_message)

    if user_message.strip()in ['EXIT','QUIT', 'BYE']:
        break
    config={'configurable':{'thread_id':thread_id}}
    response =chatbot.invoke({'messages':[HumanMessage(content=user_message)]}, config=config)

    print ("AI - ",response['messages'][-1].content)

User -  Hey, i am Neelansh.
AI -  Hello Neelansh! How can I assist you today?
User -  what is my name?
AI -  Your name is Neelansh! How can I assist you today?
User -  what is 100 plus 20 ?
AI -  100 plus 20 equals 120.
User -  multiply result with 2
AI -  120 multiplied by 2 equals 240.
User -  EXIT
